# Hospital / Emergency Services Data

## Loading Hospital Data: Filtering to NY/NJ/CT

In [1]:
import pandas as pd
import requests

hospitals = pd.read_csv(
    "Hospital_General_Information.csv"
)
ct_towns = pd.read_csv("ct_town_crosswalk.csv")

hospitals = hospitals[
    hospitals["State"].isin(["NY", "NJ", "CT"])
]

## Getting FIPS Code

In [2]:
fips_df = pd.read_csv("ny_nj_ct_fips.csv")

In [3]:
hospitals.columns = hospitals.columns.str.lower()
fips_df.columns = fips_df.columns.str.lower()

In [4]:
hospitals["county"] = hospitals["county/parish"].str.upper().str.strip()
hospitals["state"] = hospitals["state"].str.upper().str.strip()

fips_df["county"] = fips_df["county"].str.upper().str.strip()
fips_df["state"] = fips_df["state"].str.upper().str.strip()

In [5]:
fips_df["county"] = (
    fips_df["county"]
    .str.upper()
    .str.replace(" COUNTY", "", regex=False)
    .str.strip()
)

In [6]:
fips_df["fips"] = fips_df["fips"].astype(str).str.zfill(5)

In [7]:
hospitals["state"] = hospitals["state"].str.upper().str.strip()
fips_df["state"] = fips_df["state"].str.upper().str.strip()

### Merging Hospitals Dataset with FIPS Dataframe -> Creating FIPS Column

In [8]:
hospitals_geo = hospitals.merge(
    fips_df,
    on=["state", "county"],
    how="left"
)

### Getting Hospital Count Number By County

In [9]:
hospital_counts = hospitals_geo.groupby("fips").size().reset_index(name="hospital_count")

In [10]:
hospital_counts

,fips,hospital_count
0,09001,8
1,09003,8
2,09005,2
3,09007,3
4,09009,9
...,...,...
80,36113,1
81,36117,1
82,36119,12
83,36121,1


### Finding Connecticut County Population and Calculating Hospitals/100k

In [11]:
ct_towns

,town_name,town_fips_2020,county_fips,county_name,town_fips_2022,region_fips,region_name
0,Andover,901301080,9013,Tolland County,911001080,9110,Capitol Planning Region
1,Ansonia,900901220,9009,New Haven County,914001220,9140,Naugatuck Valley Planning Region
2,Ashford,901501430,9015,Windham County,915001430,9150,Northeastern Connecticut Planning Region
3,Avon,900302060,9003,Hartford County,911002060,9110,Capitol Planning Region
4,Barkhamsted,900502760,9005,Litchfield County,916002760,9160,Northwest Hills Planning Region
...,...,...,...,...,...,...,...
164,Windsor Locks,900387070,9003,Hartford County,911087070,9110,Capitol Planning Region
165,Wolcott,900987560,9009,New Haven County,914087560,9140,Naugatuck Valley Planning Region
166,Woodbridge,900987700,9009,New Haven County,917087700,9170,South Central Connecticut Planning Region
167,Woodbury,900587910,9005,Litchfield County,914087910,9140,Naugatuck Valley Planning Region


In [12]:
# Load the population data
pop_df = pd.read_csv('ct_towns_pop2023.csv')

# Standardize town names for merging (uppercase to match ct_towns)
pop_df['town_name'] = pop_df['town_name'].str.upper()

# Merge with ct_towns to get county info
merged = ct_towns[['town_name', 'county_name']].merge(
    pop_df,
    on='town_name',
    how='left'
)

# Group by county and sum populations
county_pop = merged.groupby('county_name')['pop_2023'].sum().reset_index()
county_pop.columns = ['county_name', 'total_pop_2023']
county_pop = county_pop.sort_values('total_pop_2023', ascending=False)

print(county_pop)

         county_name  total_pop_2023
0   Fairfield County             0.0
1    Hartford County             0.0
2  Litchfield County             0.0
3   Middlesex County             0.0
4   New Haven County             0.0
5  New London County             0.0
6     Tolland County             0.0
7     Windham County             0.0


In [13]:
# Check for any towns that didn't match
unmatched = merged[merged['pop_2023'].isna()]['town_name'].tolist()
print("Unmatched towns:", unmatched)

Unmatched towns: ['Andover', 'Ansonia', 'Ashford', 'Avon', 'Barkhamsted', 'Beacon Falls', 'Berlin', 'Bethany', 'Bethel', 'Bethlehem', 'Bloomfield', 'Bolton', 'Bozrah', 'Branford', 'Bridgeport', 'Bridgewater', 'Bristol', 'Brookfield', 'Brooklyn', 'Burlington', 'Canaan', 'Canterbury', 'Canton', 'Chaplin', 'Cheshire', 'Chester', 'Clinton', 'Colchester', 'Colebrook', 'Columbia', 'Cornwall', 'Coventry', 'Cromwell', 'Danbury', 'Darien', 'Deep River', 'Derby', 'Durham', 'East Granby', 'East Haddam', 'East Hampton', 'East Hartford', 'East Haven', 'East Lyme', 'East Windsor', 'Eastford', 'Easton', 'Ellington', 'Enfield', 'Essex', 'Fairfield', 'Farmington', 'Franklin', 'Glastonbury', 'Goshen', 'Granby', 'Greenwich', 'Griswold', 'Groton', 'Guilford', 'Haddam', 'Hamden', 'Hampton', 'Hartford', 'Hartland', 'Harwinton', 'Hebron', 'Kent', 'Killingly', 'Killingworth', 'Lebanon', 'Ledyard', 'Lisbon', 'Litchfield', 'Lyme', 'Madison', 'Manchester', 'Mansfield', 'Marlborough', 'Meriden', 'Middlebury', 'Mi

In [14]:
# Normalize both to uppercase for matching
ct_towns_copy = ct_towns.copy()
ct_towns_copy['town_name_upper'] = ct_towns_copy['town_name'].str.upper()
pop_df['town_name_upper'] = pop_df['town_name'].str.upper()

# Merge on the normalized column
merged = ct_towns_copy[['town_name', 'town_name_upper', 'county_name']].merge(
    pop_df[['town_name_upper', 'pop_2023']],
    on='town_name_upper',
    how='left'
)

# Check unmatched
unmatched = merged[merged['pop_2023'].isna()]['town_name'].tolist()
print("Unmatched towns:", unmatched)

# Group by county
county_pop = merged.groupby('county_name')['pop_2023'].sum().reset_index()
county_pop.columns = ['county_name', 'total_pop_2023']
county_pop = county_pop.sort_values('total_pop_2023', ascending=False)
print(county_pop)

Unmatched towns: []
         county_name  total_pop_2023
0   Fairfield County          963780
1    Hartford County          898478
4   New Haven County          865717
5  New London County          268518
2  Litchfield County          186551
3   Middlesex County          166110
6     Tolland County          150906
7     Windham County          117116


In [21]:
# Get unique county_fips + county_name from ct_towns
county_fips_map = ct_towns[['county_fips', 'county_name']].drop_duplicates()

# Merge fips into county population df
ct_county_pop = county_pop.merge(
    county_fips_map,
    on='county_name',
    how='left'
)
ct_county_pop.set_index('county_fips', inplace = True)
print(ct_county_pop)

                   county_name  total_pop_2023
county_fips                                   
9001          Fairfield County          963780
9003           Hartford County          898478
9009          New Haven County          865717
9011         New London County          268518
9005         Litchfield County          186551
9007          Middlesex County          166110
9013            Tolland County          150906
9015            Windham County          117116


In [16]:
hospitals_per_100k_all = pd.read_csv('hospitals_per_100k_nj_ny.csv')

In [17]:
hospitals_per_100k_all = hospitals_per_100k_all.merge(
    ct_county_pop[['county_name', 'total_pop_2023']],
    left_on='fips',
    right_index=True,
    how='left'
)

hospitals_per_100k_all['county'] = hospitals_per_100k_all['county'].fillna(hospitals_per_100k_all['county_name'])
hospitals_per_100k_all['population'] = hospitals_per_100k_all['population'].fillna(hospitals_per_100k_all['total_pop_2023'])
hospitals_per_100k_all['hospitals_per_100k'] = hospitals_per_100k_all['hospital_count'] / hospitals_per_100k_all['population'] * 100000

hospitals_per_100k_all.drop(columns=['county_name', 'total_pop_2023'], inplace=True)
print(hospitals_per_100k_all[hospitals_per_100k_all['fips'] < 10000])

   fips  hospital_count state             county  population  \
0  9001               8   NaN   Fairfield County    963780.0   
1  9003               8   NaN    Hartford County    898478.0   
2  9005               2   NaN  Litchfield County    186551.0   
3  9007               3   NaN   Middlesex County    166110.0   
4  9009               9   NaN   New Haven County    865717.0   
5  9011               2   NaN  New London County    268518.0   
6  9013               2   NaN     Tolland County    150906.0   
7  9015               2   NaN     Windham County    117116.0   

   hospitals_per_100k  
0            0.830065  
1            0.890395  
2            1.072093  
3            1.806032  
4            1.039601  
5            0.744829  
6            1.325328  
7            1.707709  


In [18]:
# Fill NaN state with Connecticut
hospitals_per_100k_all['state'] = hospitals_per_100k_all['state'].fillna('Connecticut')

# Remove ' County' from CT rows only
hospitals_per_100k_all['county'] = hospitals_per_100k_all['county'].str.replace(' County', '', regex=False)



In [19]:
print(hospitals_per_100k_all.tail())

     fips  hospital_count     state       county  population  \
80  36113               1  New York       Warren     65467.0   
81  36117               1  New York        Wayne     90721.0   
82  36119              12  New York  Westchester    998335.0   
83  36121               1  New York      Wyoming     39750.0   
84  36123               1  New York        Yates     24367.0   

    hospitals_per_100k  
80            1.527487  
81            1.102281  
82            1.202001  
83            2.515723  
84            4.103911  


In [20]:
hospitals_per_100k_all.to_csv('hospitals_per_100k_all.csv', index=False)